In [0]:
from pyspark.sql import functions as F

silver_maintenance = spark.table(
    "workspace.transportation_analytics.silver_maintenance_records"
)

silver_trucks = spark.table(
    "workspace.transportation_analytics.silver_trucks"
)

silver_maintenance.printSchema()
silver_trucks.printSchema()

root
 |-- maintenance_id: string (nullable = true)
 |-- truck_id: string (nullable = true)
 |-- maintenance_date: date (nullable = true)
 |-- maintenance_type: string (nullable = true)
 |-- odometer_reading: integer (nullable = true)
 |-- labor_hours: double (nullable = true)
 |-- labor_cost: double (nullable = true)
 |-- parts_cost: double (nullable = true)
 |-- total_cost: double (nullable = true)
 |-- facility_location: string (nullable = true)
 |-- downtime_hours: double (nullable = true)
 |-- service_description: string (nullable = true)

root
 |-- truck_id: string (nullable = true)
 |-- unit_number: integer (nullable = true)
 |-- make: string (nullable = true)
 |-- model_year: integer (nullable = true)
 |-- vin: string (nullable = true)
 |-- acquisition_date: date (nullable = true)
 |-- acquisition_mileage: integer (nullable = true)
 |-- fuel_type: string (nullable = true)
 |-- tank_capacity_gallons: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- home_termin

In [0]:
silver_utilization = spark.table(
    "workspace.transportation_analytics.silver_truck_utilization_metrics"
)

gold_maintenance_analysis = (
    silver_maintenance
    .join(
        silver_utilization.select(
            "truck_id",
            "total_miles"
        ),
        on="truck_id",
        how="left"
    )
)

display(gold_maintenance_analysis)

truck_id,maintenance_id,maintenance_date,maintenance_type,odometer_reading,labor_hours,labor_cost,parts_cost,total_cost,facility_location,downtime_hours,service_description,total_miles
TRK00068,MAINT00000024,2022-01-09,Preventive,439480,4.3,477.14,2446.27,2923.41,Kansas City,8.5,Routine Preventive,48742
TRK00025,MAINT00000038,2022-01-12,Repair,248118,2.7,286.6,1636.24,1922.84,Omaha,23.4,Scheduled Repair,44292
TRK00006,MAINT00000050,2022-01-17,Repair,615274,2.8,343.6,1866.29,2209.89,Salt Lake City,3.4,Emergency Repair,21162
TRK00016,MAINT00000127,2022-02-15,Brake,261631,1.2,104.59,3407.95,3512.54,Miami,22.0,Routine Brake,45378
TRK00092,MAINT00000160,2022-03-01,Repair,611879,0.7,68.09,3306.31,3374.4,Miami,32.5,Emergency Repair,26737
TRK00062,MAINT00000203,2022-03-19,Brake,441974,1.0,119.64,1977.05,2096.69,Columbus,45.2,Scheduled Brake,29727
TRK00004,MAINT00000266,2022-04-10,Engine,505751,2.5,292.9,169.2,462.1,Milwaukee,31.6,Scheduled Engine,31620
TRK00054,MAINT00000267,2022-04-10,Brake,413476,0.6,58.25,303.57,361.82,Indianapolis,40.8,Routine Brake,32903
TRK00024,MAINT00000271,2022-04-12,Preventive,177146,4.9,584.66,2567.63,3152.29,Philadelphia,4.3,Emergency Preventive,28366
TRK00060,MAINT00000278,2022-04-14,Inspection,676931,7.9,744.32,47.48,791.8,Atlanta,35.5,Routine Inspection,33754


In [0]:
truck_mileage = (
    silver_utilization
    .groupBy("truck_id")
    .agg(
        F.sum("total_miles").alias("total_miles")
    )
)

display(truck_mileage)

truck_id,total_miles
TRK00001,1356397
TRK00004,1309547
TRK00007,1336736
TRK00008,1296410
TRK00009,1310170
TRK00010,1278138
TRK00012,1341215
TRK00015,1386604
TRK00018,1194175
TRK00019,1340729


In [0]:
maintenance_by_truck = (
    silver_maintenance
    .groupBy("truck_id")
    .agg(
        F.count("maintenance_id").alias("maintenance_events"),
        F.sum("total_cost").alias("total_maintenance_cost"),
        F.sum("downtime_hours").alias("total_downtime_hours"),
        F.sum("labor_cost").alias("total_labor_cost"),
        F.sum("parts_cost").alias("total_parts_cost")
    )
)

display(maintenance_by_truck)

truck_id,maintenance_events,total_maintenance_cost,total_downtime_hours,total_labor_cost,total_parts_cost
TRK00102,23,38928.81,567.5,11463.550000000003,27465.26
TRK00068,26,48803.94000000001,632.2999999999998,10335.359999999999,38468.579999999994
TRK00025,23,44082.119999999995,570.1,10422.859999999997,33659.259999999995
TRK00006,26,50780.829999999994,587.7000000000002,10947.640000000001,39833.19
TRK00037,21,41154.92,582.6,8728.29,32426.629999999997
TRK00022,26,48894.749999999985,592.1000000000001,13589.91,35304.84
TRK00016,18,36448.05,437.09999999999997,5685.76,30762.29
TRK00092,24,41671.119999999995,433.59999999999997,11562.13,30108.989999999994
TRK00021,21,38803.04,641.1,9692.279999999999,29110.76
TRK00062,31,59646.64000000001,862.1,12240.739999999998,47405.899999999994


In [0]:
gold_maintenance_analysis = (
    maintenance_by_truck
    .join(
        truck_mileage,
        on="truck_id",
        how="left"
    )
)

display(gold_maintenance_analysis)

truck_id,maintenance_events,total_maintenance_cost,total_downtime_hours,total_labor_cost,total_parts_cost,total_miles
TRK00025,23,44082.119999999995,570.1,10422.859999999997,33659.259999999995,1370297
TRK00016,18,36448.05,437.09999999999997,5685.76,30762.29,1349432
TRK00098,18,41318.950000000004,460.29999999999995,7643.880000000001,33675.07000000001,1307617
TRK00064,22,46296.36,510.20000000000005,9158.47,37137.89,1296870
TRK00103,29,54986.22,747.1999999999999,12559.45,42426.77,1249776
TRK00020,27,56990.94999999999,711.8000000000001,10470.240000000002,46520.71,1356271
TRK00027,24,44230.56,599.3000000000001,9760.19,34470.37000000001,null
TRK00017,20,46265.84,510.00000000000006,8625.81,37640.03,null
TRK00088,26,66959.88,707.5,12660.87,54299.01000000002,null
TRK00080,21,44894.619999999995,504.3999999999999,6978.39,37916.229999999996,1268474


In [0]:
gold_maintenance_analysis = (
    gold_maintenance_analysis
    .withColumn(
        "maintenance_cost_per_mile",
        F.when(
            F.col("total_miles") > 0,
            F.round(
                F.col("total_maintenance_cost") / F.col("total_miles"),
                4
            )
        )
    )
)

display(
    gold_maintenance_analysis.select(
        "truck_id",
        "maintenance_events",
        "total_maintenance_cost",
        "total_downtime_hours",
        "total_miles",
        "maintenance_cost_per_mile"
    )
)

truck_id,maintenance_events,total_maintenance_cost,total_downtime_hours,total_miles,maintenance_cost_per_mile
TRK00025,23,44082.119999999995,570.1,1370297,0.0322
TRK00016,18,36448.05,437.09999999999997,1349432,0.027
TRK00098,18,41318.950000000004,460.29999999999995,1307617,0.0316
TRK00064,22,46296.36,510.20000000000005,1296870,0.0357
TRK00103,29,54986.22,747.1999999999999,1249776,0.044
TRK00020,27,56990.94999999999,711.8000000000001,1356271,0.042
TRK00027,24,44230.56,599.3000000000001,null,null
TRK00017,20,46265.84,510.00000000000006,null,null
TRK00088,26,66959.88,707.5,null,null
TRK00080,21,44894.619999999995,504.3999999999999,1268474,0.0354


In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "workspace.transportation_analytics.gold_maintenance_analysis"
)

target.alias("t").merge(
    gold_maintenance_analysis.alias("s"),
    "t.truck_id = s.truck_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("Gold Maintenance Analysis table updated using MERGE")

Gold Maintenance Analysis table updated using MERGE


In [0]:
display(
    spark.table(
        "workspace.transportation_analytics.gold_maintenance_analysis"
    )
)

truck_id,maintenance_events,total_maintenance_cost,total_downtime_hours,total_labor_cost,total_parts_cost,total_miles,maintenance_cost_per_mile
TRK00025,23,44082.119999999995,570.1,10422.859999999997,33659.259999999995,1370297,0.0322
TRK00016,18,36448.05,437.09999999999997,5685.76,30762.29,1349432,0.027
TRK00098,18,41318.950000000004,460.29999999999995,7643.880000000001,33675.07000000001,1307617,0.0316
TRK00064,22,46296.36,510.20000000000005,9158.47,37137.89,1296870,0.0357
TRK00103,29,54986.22,747.1999999999999,12559.45,42426.77,1249776,0.044
TRK00020,27,56990.94999999999,711.8000000000001,10470.240000000002,46520.71,1356271,0.042
TRK00027,24,44230.56,599.3000000000001,9760.19,34470.37000000001,null,null
TRK00017,20,46265.84,510.00000000000006,8625.81,37640.03,null,null
TRK00088,26,66959.88,707.5,12660.87,54299.01000000002,null,null
TRK00080,21,44894.619999999995,504.3999999999999,6978.39,37916.229999999996,1268474,0.0354


In [0]:
display(
    gold_maintenance_analysis
    .orderBy(F.desc("maintenance_cost_per_mile"))
    .select(
        "truck_id",
        "total_maintenance_cost",
        "total_miles",
        "maintenance_cost_per_mile"
    )
)

truck_id,total_maintenance_cost,total_miles,maintenance_cost_per_mile
TRK00014,71801.81999999999,1240298,0.0579
TRK00040,67557.73999999999,1178515,0.0573
TRK00073,76573.98999999998,1338563,0.0572
TRK00099,73231.77,1291808,0.0567
TRK00026,70708.48,1252565,0.0565
TRK00038,68989.13999999998,1262396,0.0546
TRK00033,67514.42999999996,1274427,0.053
TRK00052,63757.21000000001,1241107,0.0514
TRK00044,69825.23,1403922,0.0497
TRK00071,61696.55,1269268,0.0486


Databricks visualization. Run in Databricks to view.

In [0]:
display(
    gold_maintenance_analysis
    .orderBy(F.desc("total_downtime_hours"))
    .select(
        "truck_id",
        "total_downtime_hours",
        "total_maintenance_cost"
    )
)

truck_id,total_downtime_hours,total_maintenance_cost
TRK00003,1133.1000000000001,90161.42
TRK00044,1080.3999999999999,69825.23
TRK00073,996.2999999999998,76573.98999999998
TRK00046,974.5,57517.479999999996
TRK00014,940.4000000000001,71801.81999999999
TRK00118,917.2999999999998,71005.76999999999
TRK00033,908.1999999999999,67514.42999999996
TRK00013,868.5,57951.159999999996
TRK00062,862.1,59646.64000000001
TRK00078,839.6000000000003,48928.869999999995


Databricks visualization. Run in Databricks to view.